# SPICE — Phase 2 Post-train / RL (Colab)

> Two paths: Path A (SAC micro) + Path B (ES macro) + quick screening / phase maps / pseudo-label reflow, backed by `spice_engine`.
>
> **Prerequisite**: `spice_engine` is compiled into a **whl**; this notebook installs it in step ②.
>
> Run order: ① deps → ② engine → ③ load Pre-train artifacts → ④ run two-path RL → ⑤ fine-tune → ⑥ visualize.

In [ ]:
# ① Dependencies (China pip mirror) + check GPU
# 版本钉死：与保存 checkpoint 的环境一致（TF 2.21 + Keras 3.15）。
# 不同 Keras 版本的优化器 checkpoint 布局/dtype 不同（step_counter int64→float32 等），
# 跨平台续训会 RestoreV2 报错；钉死版本后 optimizer 状态即可完整恢复。
# 注意：Kaggle 上清华镜像可能不通——跑不动就把 "-i https://pypi.tuna.tsinghua.edu.cn/simple" 去掉用默认 PyPI。
%pip install -q -i https://pypi.tuna.tsinghua.edu.cn/simple \
    "tensorflow==2.21" "keras==3.15" "tensorboard==2.21" \
    datasets huggingface_hub pyarrow polars pyyaml matplotlib tqdm

# RL 阶段是 MD 瓶颈（spice_engine 纯 CPU），模型/SAC 更新毫秒级 → 纯 CPU 跑，不占 GPU 配额。
# 本 notebook 只做 Phase 2（RL），无需 GPU。想用 GPU 就注释掉下面这行。
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf
import keras
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("Device: forced CPU")

## ② Install spice_engine whl

`spice_engine` is a PyO3 library compiled from the local `md_cal` (Linux whl). **Choose one**:
- Option A: upload the `.whl` file to the Colab root directory
- Option B: put a direct whl URL in the code below

In [ ]:
import glob, subprocess, sys

# ⚠️ wheel 的 cp 标签必须匹配运行时 Python：Colab=3.12→cp312；下面会打印当前版本。
# 上传的 whl 若是 cp39/cp311 会装不上（pip: not a supported wheel on this platform）。
runtime = f"cp{sys.version_info.major}{sys.version_info.minor}"
pyver = f"{sys.version_info.major}.{sys.version_info.minor}"
print("Runtime Python:", sys.version.split()[0], f"(tag {runtime})")

def check_and_install(path):
    parts = path.split("-")
    if "abi3" in parts:
        pass  # abi3 稳定 ABI，跨 Python 版本可用
    else:
        tags = [p for p in parts if p.startswith("cp")]
        if tags and tags[0] != runtime:
            raise SystemExit(
                f"wheel Python 标签 {tags[0]} ≠ 运行时 {runtime}，pip 会拒绝安装。\n"
                f"请在 Linux/Docker 里用目标 Python 重建（manylinux 只能在 Linux 打）：\n"
                f"  maturin build --release --interpreter python{pyver}\n"
                f"然后把 cp{pyver.replace('.', '')} 的 .whl 传上来。")
    print("Installing:", path)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", path])

# Option A: whl uploaded to this directory
wheels = sorted(glob.glob("*.whl"))
if wheels:
    check_and_install(wheels[0])

# Option B: direct URL (uncomment and fill in the address)
# url = "https://your-host/spice_engine-0.1.0-cp312-cp312-manylinux_2_17_x86_64.whl"
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", url])

import spice_engine
print("spice_engine OK, version:", spice_engine.version())

## ③ Load Pre-train artifacts (verification)

The three artifacts produced by Pre-train (`spice_pre`) are all reused by RL:

| Pre-train artifact | Purpose | Consumed in RL by |
|---|---|---|
| `checkpoints/pretrain/best_weights.weights.h5` | backbone + Head A weights | `train_post.build_rl_model` loads with `skip_mismatch=True` (Head B/C/D are random, left to ES/SAC) |
| `data/tfrecords/shard_*.tfrecord` | pre-training data | `finetune_pretrain` merges with the original data when reflowing |
| `configs/pretrain.yaml` | model-structure hyperparameters (embed_dim, etc.) | `build_rl_model` reads the model structure to stay consistent with Pre-train |

These correspond to the `pretrain_ckpt` / `pretrain_tfrecord_dir` / `pretrain_config` pointers in `configs/posttrain.yaml`. The cell below verifies loading:

In [ ]:
from spice_pre.config import load_config as pre_load
from spice_pre.models import SPICEPretrainModel
from spice_rl.config import load_config as rl_load

rl_cfg = rl_load("configs/posttrain.yaml")
pre_cfg = pre_load(rl_cfg.post.pretrain_config)   # read the Pre-train model structure

# Build the full two-path model (all heads enabled)
import numpy as np, tensorflow as tf
model = SPICEPretrainModel(pre_cfg.model, heads=("A", "B", "Bp", "C", "D"))
model({"tokens": tf.zeros([1,8], tf.int32), "env": tf.zeros([1,3]), "mask": tf.ones([1,8])}, training=False)

import os
if os.path.exists(rl_cfg.post.pretrain_ckpt):
    model.load_weights(rl_cfg.post.pretrain_ckpt, skip_mismatch=True)
    print("Loaded Pre-train weights (backbone + Head A)")
print("All five heads present:", all(getattr(model, h) is not None for h in
      ("head_a","head_b","head_bp","head_c","head_d")))
print("embed_dim:", model.embed_dim)

## ④ Run two-path RL (requires the engine)

`train_post` 主循环 = 设计文档四环节（环节一·预训练已在上游完成）：

1. **环节二·快筛**：全原子 MD 短跑（20 步）仅查物理合法性（build 成功 + 不崩溃/无能量爆炸），违规淘汰
2. **环节三·路径 A（不突变）**：固定序列，一正一反双线程扰动环境（+ΔpH/+ΔT 与 −ΔpH/−ΔT，实现为顺序执行），SAC 微观环施偏置力探稳定区间/崩溃边界；周期扫 pH-T 相图
3. **环节三·路径 B（可能突变）**：**仅当路径 A 崩溃（记录 `Env_fail`）才触发**；ES 宏观环在 Head-B 采样 1~3 点突变，**冻结复用路径 A 的 SAC Actor** 评估存活步数
4. **环节四·双路回流**：路径 A 崩溃 → 负样本 `Env_fail`；路径 B 存活 → 时间平均坐标伪标签回流

**Cell ④ 拆两段**：

- **④a 数据准备 + 选亲本**（只跑一次）：下载全量 entries 元数据 → 筛 ≤150aa → 分层随机选候选 `shortlist`。`SELECT_MODE`：`stratified`=生产（60–150 aa 的 X-ray，优先真实 env）／`shortest`=调试（避免永远挑最短抽到 poly-K 类退化短肽）。
- **④b 快筛重试 + 跑 RL**（可反复重跑）：逐个候选下载 atoms shard → 加载全原子结构 → 快筛 → 通过的进 `train_post`。引擎力场覆盖不到的修饰残基/几何会让 `Engine.build` 失败（如 5DKQ `XC-N-XC`）→ 自动跳过换下一个（`N_TRIES=5`）。

初始结构从数据管线 parquet 加载（**全原子** → `from_atoms`，引擎自动加氢，无需 mmCIF；CA 重建已否决）；**只收 ≤ `post.max_seq_len`(=150) aa 的短蛋白**（超长会截断导致序列↔坐标错位）。参数在 `configs/posttrain.yaml`（可用 `--max-episodes` 覆盖；下方为调试用少量 episode）。

In [ ]:
# ④a 数据准备 + 选亲本（只跑一次；候选 shortlist 供 ④b 使用）
rl_cfg.env.relax_iters = 100
rl_cfg.post.max_episodes = 30   # 完整训练集数（yaml 默认 30；<5 集 SAC 只收集不学习，buffer 到 200 才首次更新）

# ---- 环境扰动幅度（在 cell 里覆盖，不用改 yaml 文件）----
# 稳定亲本 + 温和扰动(0.5pH/5K) = 路径 A 永不崩溃 = 路径 B 不触发 = 没有伪标签 = finetune 空跳过。
# 调大让路径 A 撞上崩溃边界 → 触发路径 B → 存活突变体写伪标签 → finetune 才有回流数据。
rl_cfg.post.env_delta_ph = 5.0     # ±5 pH（扰动环境 pH2/12，m5 表面电荷错配才够强 → 触发路径 B）
rl_cfg.post.env_delta_T  = 25.0    # ±25 K（yaml 默认 5.0；同左）
# 观察日志：出现 "[ep N] 路径A 崩溃" 才有路径 B；"存活突变体 n" > 0 才会写伪标签。
# ④b 若报"全部 margin≈1.0（超稳）"：先加大候选池（N_TRIES 上调），还不够才继续调大 delta。
# 全崩（无存活）→ 回调一点；全不崩 → 再调大。

import os, random
# HF 端点自动选择：
#   Colab / Kaggle：云端，官方 huggingface.co 直连即可（Kaggle 也走官方，无需镜像）
#   本地（国内）：官方被墙 → 用 hf-mirror.com
try:
    import google.colab  # noqa: F401
    is_cloud = True
except ImportError:
    is_cloud = os.path.isdir("/kaggle")     # Kaggle notebook 特征路径
endpoint = "https://huggingface.co" if is_cloud else "https://hf-mirror.com"
os.environ["HF_ENDPOINT"] = endpoint
import huggingface_hub.constants as hf_constants
hf_constants.ENDPOINT = endpoint   # huggingface_hub 只在首次 import 时读 env，必须直接改 constants
print("HF endpoint:", endpoint)

# 全量扫描：把全部 entries shard（序列元数据，很小）拉下来，全局筛 ≤ max_seq_len aa。
# （只下载选中蛋白的 atoms shard 在 ④b 做，重原子全量 ~20GB 不整包拉。）
from huggingface_hub import hf_hub_download, list_repo_files
import polars as pl
REPO = "SPICE-Protein/spice_protein"
entries = sorted(f for f in list_repo_files(REPO, repo_type="dataset")
                 if f.startswith("entries_shard_") and f.endswith(".parquet"))
print(f"共 {len(entries)} 个 entries shard，逐个下载并筛选…")
frames = []
for f in entries:
    en = pl.read_parquet(hf_hub_download(REPO, f, repo_type="dataset"))
    frames.append(en.with_columns(pl.lit(f).alias("_src")))
en_all = pl.concat(frames)
cands = en_all.filter(pl.col("n_residues") <= rl_cfg.post.max_seq_len)
print(f"全量 ≤{rl_cfg.post.max_seq_len} aa 的蛋白: {cands.height} 个")
if cands.height == 0:
    raise SystemExit(f"全量数据没有 ≤{rl_cfg.post.max_seq_len} aa 的蛋白？调大 max_seq_len 看看")

# ---- 候选池（stratified=生产 / shortest=调试）----
# 不要永远选最短：会抽到 poly-K 之类退化肽段（无折叠可优化、无突变信息量）。
# 分层：min_seq_len~max_seq_len aa 的结构（跳过 <min_seq_len aa 柔性小肽——太短无折叠可学、突变泛化差），
#       优先真实 env（env 条件化实验），没有再回退无 env。
# 低分辨率优先：低分辨率结构通常更"松"、稳定性更边缘 → 更可能触发路径 B。
SELECT_MODE = "stratified"   # "stratified"=生产（min_seq_len+ aa，低分辨率优先）| "shortest"=调试（最短蛋白）
N_TRIES = 10                 # 最多试 N 个候选（④b 边缘筛选用；全超稳时上调，多筛才容易撞到边缘蛋白）
MIN_AA = rl_cfg.post.min_seq_len   # 最短残基数（yaml min_seq_len=80；<80aa 多为无折叠小肽）
if SELECT_MODE == "shortest":
    shortlist = cands.sort("n_residues").head(N_TRIES).to_dicts()
else:
    pool = cands.filter(pl.col("n_residues") >= MIN_AA)
    # ⚠️ 实测：好 X-ray 结构在引擎里几乎不 blow-up（2LYZ 400K/pH2-13 都稳）。
    # 崩溃主要来自构建质量差的蛋白 → 放宽方法过滤（NMR/EM/低质量更可能 blow-up）。
    # pool = pool.filter(pl.col("method").str.contains("X-RAY"))   # (可选收紧回 X-ray)
    if pool.height == 0:                     # 防御：池空则放宽到全长度
        pool = cands.filter(pl.col("n_residues") >= MIN_AA)
    if pool.height == 0:
        pool = cands
    pool_env = pool.filter(pl.col("has_env"))
    print(f"候选池({MIN_AA}-{rl_cfg.post.max_seq_len} aa, all methods): {pool.height} | 含真实env: {pool_env.height}")
    pick_from = pool_env if pool_env.height > 0 else pool
    # 低分辨率优先：按 resolution 升序，从前 20 个里抽候选
    pick_from = pick_from.with_columns(pl.col("resolution").fill_null(float("inf")))
    pick_from = pick_from.sort("resolution")
    shortlist = pick_from.head(max(20, pick_from.height)).sample(n=min(N_TRIES, pick_from.height), seed=random.randrange(10 ** 9)).to_dicts()
    print("候选 shortlist:", [(r["pdb_id"], r["n_residues"]) for r in shortlist])

In [ ]:
# ④b pH/m5 不稳定筛选 + 跑 RL（可反复重跑；候选池来自 ④a 的 shortlist）
# 目标：挑"能折（build 合法）但在极端 pH 下表面电荷错配(m5)明显恶化"的蛋白——
# 这类才是路径 B 该探索的（pH 失稳 → 找恢复电荷平衡的更稳突变构象）。
# 超稳蛋白（极端 pH 下 m5 无响应，margin≈1.0）pass：不强行突变。
# 引擎 build 失败（力场覆盖不到/侧链不完整）自动跳过。
#
# ⚠️ 为什么用 pH/m5 而不是温度：温度轴经 equilibrate 修复后虽真实，但短窗只能看到
#    去折叠开端（Q 缓慢掉，400 步才 -6%），信号弱；m5（表面电荷错配）在极端 pH 强且
#    干净（实测 2LYZ: pH7=0.16 → pH2=0.25，+54% 低方差），且 pH 轴正是论文押注的 novelty。
#
# ⚠️ 提速（2026-08-11）：探针跳 equilibrate（margin 是 m5 电荷信号，不需要 400 步温度
#    平衡，探针成本 -86%）。
#
# ⚠️ 筛选带（2026-08-11 两次实跑教训：6I75/6AC5 都 margin≈0，只在 pH 地板崩=不可救）：
#    - STABLE_CUTOFF=0.95：margin≥0.95（比率<~1.4× 噪声）→ 太稳，跳过
#    - TOO_UNSTABLE=0.1：margin<0.1（比率>~2.8×）→ 只在极端 pH 才崩，救援目标砸地板=不可救，跳过
#    - MIN_EDGE_COUNT=3：集齐 ≥3 个边缘候选再训，最脆的先试、后几个作后备（0 存活 abort 后自动跳）
#    可接受带 = [0.1, 0.95)，即"能温和崩"的蛋白。
import os, shutil, inspect
from huggingface_hub import hf_hub_download
from spice_rl.env import load_structure_with_atoms, quick_check_env
from spice_rl.train_post import train

MARGIN_PROBE_STEPS = 60      # 探针窗口（步；anchor/pH2/pH12 都用它，等长）
PH_PROBE_LOW  = 2.0          # 极端低 pH 探针
PH_PROBE_HIGH = 12.0         # 极端高 pH 探针
M5_EDGE_LO = 1.3             # m5 比率 ≥ 1.3×anchor = 开始 pH 失稳（margin 开始降）
M5_EDGE_HI = 3.0             # m5 比率 ≥ 3×anchor = 强 pH 失稳（margin→0）
STABLE_CUTOFF = 0.95         # margin ≥0.95（m5 比率<~1.4×，噪声级）= 太稳，跳过（不进候选）
TOO_UNSTABLE  = 0.1          # margin <0.1（m5 比率>~2.8×）= 只在极端 pH 才崩→救援目标砸地板=不可救，跳过
MIN_EDGE_COUNT = 3           # 至少集齐 N 个边缘候选再训（top-K 有后备；0 存活 abort 后能跳下一个）
# 探针跳 equilibrate：m5 电荷信号不需要温度平衡（探针成本 -86%）。
# 旧 quick_check 无 equilibrate 参数时回退（照常平衡，慢但正确）。
_PROBE_KW = {"equilibrate": False} if "equilibrate" in inspect.signature(quick_check_env).parameters else {}


def margin_of(struct):
    """anchor(60步) 测 m5 基线 + pH2/pH12 极端探针：margin 由 m5 比率定义（越低越 pH 边缘）。"""
    a = quick_check_env(struct, rl_cfg.env, rl_cfg.post.anchor_ph, rl_cfg.post.anchor_temp,
                        n_steps=MARGIN_PROBE_STEPS, **_PROBE_KW)
    if not a["ok"]:
        return None, a["reason"], a
    anchor_m5 = a.get("m5_mean") or 0.0
    p_low = quick_check_env(struct, rl_cfg.env, PH_PROBE_LOW, rl_cfg.post.anchor_temp,
                            n_steps=MARGIN_PROBE_STEPS, **_PROBE_KW)
    if not p_low["ok"]:
        return None, p_low["reason"], a
    p_high = quick_check_env(struct, rl_cfg.env, PH_PROBE_HIGH, rl_cfg.post.anchor_temp,
                             n_steps=MARGIN_PROBE_STEPS, **_PROBE_KW)
    if not p_high["ok"]:
        return None, p_high["reason"], a
    m5_pert = max(p_low.get("m5_mean") or 0.0, p_high.get("m5_mean") or 0.0)
    ratio = m5_pert / max(anchor_m5, 1e-6) if anchor_m5 > 0 else 0.0
    # margin: ratio≤M5_EDGE_LO → 1.0(稳,pass)；ratio≥M5_EDGE_HI → 0(边缘,路径 B 目标)
    margin = max(0.0, min(1.0, 1.0 - (ratio - M5_EDGE_LO) / (M5_EDGE_HI - M5_EDGE_LO)))
    return margin, "ok", a


os.makedirs("/content/parquet", exist_ok=True)
screened = []            # 通过：真边缘候选（TOO_UNSTABLE ≤ margin < STABLE_CUTOFF）
skipped_stable = 0       # 跳过：噪声级弱响应（margin ≥ STABLE_CUTOFF）
skipped_fragile = 0      # 跳过：只在极端 pH 才崩（margin < TOO_UNSTABLE）
n_tried = 0
for row in shortlist:
    n_tried += 1
    PDB_ID = row["pdb_id"]
    atoms_src = row["_src"].replace("entries_shard_", "atoms_shard_")
    # ⚠️ 防御：太短无折叠可学（与 ④a 池过滤双保险；shortest 调试模式也会被拦）
    if row["n_residues"] < rl_cfg.post.min_seq_len:
        print(f"  跳过 {PDB_ID} (n_residues={row['n_residues']} < min_seq_len={rl_cfg.post.min_seq_len}，太短无折叠可学)")
        continue
    try:
        print("尝试:", PDB_ID,
              f"(n_residues={row['n_residues']}, has_env={bool(row['has_env'])}) | {atoms_src}")
        shutil.copy(hf_hub_download(REPO, atoms_src, repo_type="dataset"),
                    os.path.join("/content/parquet", os.path.basename(atoms_src)))
        struct, base_atoms = load_structure_with_atoms(
            "/content/parquet", PDB_ID, max_residues=rl_cfg.post.max_seq_len)
        seq = struct.sequence()
        margin, reason, a = margin_of(struct)
        if margin is None:
            print(f"  未通过（{reason}），跳过")
            continue
        if margin >= STABLE_CUTOFF:
            # margin≈1.0 = m5 响应噪声级（如 0.97 只是 ~1.35×，接近测量噪声）→ 太稳，
            # path A 不会触发 Env_fail → path B 空转 → 白跑一整个 RL。跳过不入选。
            skipped_stable += 1
            print(f"  跳过: margin={margin:.3f} ≥ {STABLE_CUTOFF}（m5 响应太弱≈稳定，非路径 B 目标）")
            continue
        if margin < TOO_UNSTABLE:
            # margin<0.1 = 只在极端 pH 才崩（6I75/6AC5 教训）→ 路径 B 救援目标必然砸 pH 地板，
            # 单个突变救不回 → 白跑。跳过不入选。
            skipped_fragile += 1
            print(f"  跳过: margin={margin:.3f} < {TOO_UNSTABLE}（只在地板崩=不可救，非路径 B 目标）")
            continue
        print(f"  通过: margin={margin:.3f} (m5 比率, U_anchor={a['u']:.1f}) → 进边缘候选")
        screened.append((margin, PDB_ID, struct, base_atoms, seq, row))
        if len(screened) >= MIN_EDGE_COUNT:
            print(f"  已集齐 {len(screened)} 个边缘候选 → 停止筛选（不再筛剩余 {len(shortlist)-n_tried} 个）")
            break
    except Exception as e:  # noqa: BLE001
        print(f"  候选 {PDB_ID} 处理失败: {e}，跳过")
print(f"筛选完成: 通过边缘候选 {len(screened)} 个 | 跳过稳定 {skipped_stable} 个 | 跳过太脆(极端) {skipped_fragile} 个")
if not screened:
    raise SystemExit(
        f"{len(shortlist)} 个候选全部未通过（build 失败/不合法/过短/太稳/太脆）。"
        f"边缘候选需 {TOO_UNSTABLE} ≤ margin < {STABLE_CUTOFF}（m5 比率 ~1.4~2.8×）。"
        f"可把 PH_PROBE 再调极端（如 1.0/13.0）、放宽 TOO_UNSTABLE、调大 N_TRIES、或换 SELECT_MODE/shortest 试试")

# ---- 只保留有真 pH/m5 响应的蛋白；超稳/太脆已在上一步拦掉 ----
# TOO_UNSTABLE ≤ margin < STABLE_CUTOFF = 极端 pH 下 m5 显著恶化、但能温和崩 → 路径 B 目标。
edge = screened   # 已过滤：全部是真边缘候选
edge.sort(key=lambda x: x[0])   # margin 升序 = 越边缘（越脆）越靠前先试；后几个作后备

# ---- 多蛋白：逐个跑 RL（前几个最脆，后几个后备；0 存活 abort 后自动跳下一个）----
# 每个蛋白产伪标签 → 共享 data/pseudo_labels/（文件名带 PDB tag 防覆盖）→ ⑤ 统一微调。
# 多蛋白 = 伪标签多样性（不同折叠 + 不同 env）→ Head A 学通用折叠，不背单蛋白。
# ⚠️ 一个蛋白失败（try/except）不中断整个循环；算力线性增长，K 越大越多样但越慢。
TOP_K = 3                       # 跑几个边缘蛋白（默认 3，可调；越大伪标签越多样但越慢）
print(f"边缘候选 {len(edge)} 个（margin 排序），取 top-{min(TOP_K, len(edge))} 个逐个跑 RL")
for _k, (margin, PDB_ID, struct, base_atoms, seq, row) in enumerate(edge[:TOP_K]):
    print(f"=== [{_k+1}/{min(TOP_K, len(edge))}] 边缘蛋白 {PDB_ID} "
          f"(margin={margin:.3f}, n_res={row['n_residues']}) ===")
    try:
        train(rl_cfg, struct, seq, base_atoms=base_atoms, tag=PDB_ID.lower())
    except Exception as e:  # noqa: BLE001
        print(f"  蛋白 {PDB_ID} RL 失败（跳过，继续下一个）: {e}")
print("④b 多蛋白循环完成")

In [ ]:
# ④c Head A 折叠微调：MD 时间平均坐标 → 伪标签（不依赖预筛 / 路径 B）
# 动机：Head A 学会折叠才是 RL 的最终目的。原链路（路径 B 崩溃 → 存活突变体 → 伪标签）
#       太脆：好蛋白不崩 → 路径 B 不触发 → 伪标签空 → Head A 永远学不会折叠。
# 解法：对任何能 build 的蛋白直接跑 MD，把引擎的时间平均坐标（engine.pseudo_labels）
#       当伪标签教 Head A —— "物理当老师"最直接的形态，size-agnostic，不需要运气。
# 稳定蛋白反而最好：MD 保持折叠 → 时间平均坐标 ≈ 折叠结构的热力学平均 → 理想 3D 标签。
# ⚠️ 需先跑 ④a（拿 shortlist / REPO / rl_cfg）；⑤ 统一做回流微调。
import os, shutil
import numpy as np
from huggingface_hub import hf_hub_download
from spice_rl.env import load_structure_with_atoms
from spice_rl.env.md_env import MDSimulationEnv

# 蛋白选择：None = 用 ④a shortlist 里第一个能 build 的；或手动填 PDB_ID（如 "1UBQ"）
MD_PDB = None
EQUIL_STEPS = 20      # 平衡步（丢弃；引擎可靠窗口内）
PROD_STEPS  = 40      # 生产步：时间平均坐标来自这部分


def _load(pdb, atoms_src):
    shutil.copy(hf_hub_download(REPO, atoms_src, repo_type="dataset"),
                os.path.join("/content/parquet", os.path.basename(atoms_src)))
    return load_structure_with_atoms("/content/parquet", pdb,
                                     max_residues=rl_cfg.post.max_seq_len)


os.makedirs("/content/parquet", exist_ok=True)
struct = base_atoms = seq = None
if MD_PDB:
    row = next(r for r in shortlist if r["pdb_id"].upper() == MD_PDB.upper())
    struct, base_atoms = _load(MD_PDB.upper(), row["_src"].replace("entries_shard_", "atoms_shard_"))
    seq = struct.sequence()
else:
    for row in shortlist:
        PDB_ID = row["pdb_id"]
        try:
            struct, base_atoms = _load(PDB_ID, row["_src"].replace("entries_shard_", "atoms_shard_"))
            seq = struct.sequence()
            print(f"选中 {PDB_ID} (n_res={struct.residue_count()}) 跑 MD 生成折叠伪标签")
            break
        except Exception as e:  # noqa: BLE001
            print(f"  {PDB_ID} build 失败跳过: {e}")
if struct is None:
    raise SystemExit("没有能 build 的蛋白——先跑 ④a 选候选池")

# ---- 跑 MD，收集时间平均坐标 ----
env = MDSimulationEnv(struct, rl_cfg.env,
                      ph=rl_cfg.post.anchor_ph, temp=rl_cfg.post.anchor_temp,
                      ionic=rl_cfg.env.ionic_default, reuse_engine=True)
env.reset()
zero = np.zeros(env.act_dim, np.float32)
for _ in range(EQUIL_STEPS):          # 平衡（丢弃）
    env.step(zero)
env.reset_pseudo_labels()
for _ in range(PROD_STEPS):           # 生产（累积时间平均 Cα 坐标）
    env.step(zero)
coords = env.pseudo_labels()          # (L,3) 时间平均 Cα
env_norm = np.array([rl_cfg.post.anchor_ph, rl_cfg.post.anchor_temp,
                     rl_cfg.env.ionic_default], np.float32)
os.makedirs(rl_cfg.post.pseudo_label_dir, exist_ok=True)
fn = os.path.join(rl_cfg.post.pseudo_label_dir, f"pseudo_0_{PROD_STEPS}.npz")
np.savez(fn, seq=seq, env=env_norm, coords=coords)
print(f"MD 时间平均坐标伪标签 -> {fn}  (n_res={coords.shape[0]})")
print("伪标签就绪：跑 ⑤ 完成 Head A 微调（finetune 内部会 write_pseudo_tfrecord + 合并 Pre-train）")

## ⑤ Pseudo-label reflow → fine-tune (can run without the engine)

Time-averaged coordinates of surviving Path-B mutants (`data/pseudo_labels/pseudo_*.npz`) are reflowed with confidence weighting, merged with the original Pre-train TFRecords to fine-tune Head A → `finetuned.weights.h5`.

In [ ]:
from spice_rl.finetune_pretrain import finetune
rl_cfg.post.finetune_epochs = 1
finetune(rl_cfg)

## ⑥ Visualization: stability phase maps

Reads `runs/posttrain/phase_maps/*.npz` (the pH-T plane scanned by the engine) and plots stable/collapse boundaries. If there's no data yet, an example is shown to demonstrate the axes.

In [ ]:
import glob, numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

files = sorted(glob.glob("runs/posttrain/phase_maps/*.npz"))
if files:
    d = np.load(files[-1], allow_pickle=True)
    ph, temp = np.asarray(d["ph"]), np.asarray(d["temp"])
    stable = np.asarray(d["stable"]).astype(bool)
    crashed = np.asarray(d.get("crashed", np.zeros_like(stable))).astype(bool)
    # 标注稳定 / 崩溃边界（设计文档：pH-T 平面标注稳定与崩溃边界）
    cls = np.zeros(len(ph), dtype=int)                 # 0 = boundary（不稳定但未崩溃）
    cls[stable] = 1                                    # 1 = stable
    cls[(~stable) & crashed] = 2                       # 2 = crashed
    cmap = ListedColormap(["#f4d03f", "#27ae60", "#e74c3c"])  # 黄=边界 绿=稳定 红=崩溃
    plt.figure(figsize=(7, 5))
    sc = plt.scatter(ph, temp, c=cls, cmap=cmap, s=60, vmin=0, vmax=2)
    cb = plt.colorbar(sc, ticks=[0, 1, 2])
    cb.ax.set_yticklabels(["boundary", "stable", "crashed"])
    plt.xlabel("pH"); plt.ylabel("Temperature (K)")
    plt.title(f"Stability phase map {files[-1].split('/')[-1]}")
    plt.show()
else:
    # No data: example phase map
    ph = np.linspace(0, 14, 29); T = np.linspace(280, 340, 13)
    PH, TT = np.meshgrid(ph, T)
    stable = (TT < 320) & (PH > 3) & (PH < 11)
    plt.figure(figsize=(7,5))
    plt.pcolormesh(PH, TT, stable.astype(int), cmap="RdYlGn", shading="auto")
    plt.colorbar(label="stable")
    plt.xlabel("pH"); plt.ylabel("Temperature (K)"); plt.title("Stability phase map (example)")
    plt.show()
    print("(Example data: running train_post generates real phase maps that will be shown automatically)")

## Summary

- Pre-train artifacts (weights / TFRecords / structure config) are reused by RL via the three pointers in `configs/posttrain.yaml`.
- Two-path RL: quick screening → Path A SAC explores boundaries + phase maps → Path B ES mutation (frozen SAC-Actor evaluation) → pseudo-label reflow & fine-tune.
- Outputs: `runs/posttrain/phase_maps/*.npz` (phase maps), `data/pseudo_labels/*.npz` (pseudo-labels), `checkpoints/pretrain/finetuned.weights.h5` (fine-tuned weights).